In [1]:
words = open('../names.txt', 'r').read().splitlines()

In [2]:
occurences = {}
for word in words:
    value = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(value, value[1:]):
        data = (ch1,ch2)
        occurences[data] = occurences.get(data, 0) + 1

In [3]:
characters = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(characters)}
stoi['.'] = 0
stoi
itos = {i:s for s,i in stoi.items()}

In [7]:
import torch

In [ ]:
## Creating data set which will have a 3 characters and using that we will try to predict next character so only thing changed from last make more implementation is that 
## no we will store three character index in xs and one in ys which needs to be predicted
block_size = 3
xs = []
ys = []
for word in words[:1]:
    val = [0] * block_size
    for ch in word + '.':
        xs.append(val)
        ys.append(stoi.get(ch))
        val = val[1:] + [stoi.get(ch)]
# print(f'{xs} -> {ys}')


[[0, 0, 0], [0, 0, 5], [0, 5, 13], [5, 13, 13]] -> [5, 13, 13, 1]


In [8]:
X = torch.tensor(xs)
Y = torch.tensor(ys)

In [18]:
## lets create embedding matrix instead of one hot encoding as becuase this helps understand the relationship between characters and also they are much more expressive
## and scalable.
lookUpMat = torch.randn((27,2))
lookUpMat

tensor([[ 2.0376, -0.0589],
        [ 1.2462,  1.1005],
        [-0.6639, -1.9078],
        [ 1.6298,  1.8938],
        [-0.4133, -0.2041],
        [ 0.6286, -1.2949],
        [ 0.6375,  0.0897],
        [-0.1462, -1.1465],
        [ 0.2542, -1.0374],
        [-0.7212,  1.2268],
        [-0.1371, -0.0347],
        [ 2.1391,  0.3629],
        [ 0.2511,  0.4944],
        [ 0.7001, -0.2033],
        [-1.8351,  1.0892],
        [ 1.2919, -0.3285],
        [-0.4066,  0.7535],
        [ 0.7199, -0.1912],
        [ 2.2203,  0.4185],
        [ 2.0188,  1.9681],
        [-0.4154,  0.1002],
        [ 0.0638,  0.7014],
        [ 1.0361, -1.5250],
        [ 0.8983, -0.9761],
        [-0.1032, -0.7277],
        [-1.0402, -1.8651],
        [ 0.2874, -1.7530]])

In [20]:
emb = lookUpMat[X]

In [23]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)
W2 = torch.randn(100,27)
b2 = torch.randn(27)


In [28]:
h = torch.tanh(emb.view(-1,6) @ W1 + b1) ## activation function
logits = h @ W2 + b2
## forward pass

In [ ]:
norm = logits.exp()
pred = norm / norm.sum(dim=1, keepdim=True)
pred

tensor([[2.9795e-12, 2.3349e-09, 7.0381e-05, 5.3471e-05, 1.2537e-09, 4.0847e-05,
         1.5728e-13, 3.8514e-02, 6.6302e-13, 7.7971e-05, 7.8784e-05, 5.5963e-05,
         9.6616e-13, 3.5796e-13, 3.9972e-08, 2.8042e-02, 1.5493e-07, 3.7753e-01,
         8.8009e-11, 7.4489e-09, 4.1060e-14, 1.7980e-03, 2.2350e-08, 1.7024e-12,
         4.9019e-13, 5.5373e-01, 4.2178e-06],
        [5.8191e-14, 7.0704e-11, 5.6112e-01, 5.8236e-05, 3.9091e-09, 4.0824e-08,
         3.5452e-09, 1.0785e-02, 8.0886e-17, 1.4719e-01, 1.0463e-04, 6.3611e-07,
         1.0936e-07, 1.0042e-09, 1.5767e-10, 2.4301e-03, 7.4594e-08, 1.2726e-04,
         3.1530e-10, 1.4292e-11, 3.7199e-10, 1.5794e-05, 9.3689e-09, 1.2352e-13,
         5.8453e-11, 2.7772e-01, 4.4041e-04],
        [3.4780e-15, 1.9435e-09, 7.6907e-01, 4.3031e-08, 1.7093e-09, 7.1759e-11,
         4.1023e-11, 1.3682e-04, 4.9310e-15, 2.2787e-01, 2.4117e-03, 7.9375e-06,
         1.8874e-09, 1.5389e-13, 2.9125e-07, 4.7380e-04, 2.8550e-11, 1.1749e-08,
         3.5059e-

In [ ]:
loss = -pred[torch.arange(len(X)), Y].log().mean() # loss calculation
loss

tensor(15.7781)

In [38]:
# structured code
g = torch.Generator().manual_seed(2147483647)
lookUpMat = torch.randn((27,2), requires_grad=True)
emb = lookUpMat[X]
W1 = torch.randn((6,100), requires_grad=True)
b1 = torch.randn((100), requires_grad=True)
W2 = torch.randn((100,27), requires_grad=True)
b2 = torch.randn((27),requires_grad=True)
params = [lookUpMat, W1, b1, W2, b2]

In [34]:
sum(p.nelement() for p in params)

3481

In [39]:
for p in params:
    p.grad = None

In [ ]:
## Finding suitable learning rate


In [ ]:
## Do a batch training

In [40]:
## starting to train the model
for _ in range(30):
    ## forward pass
    emb = lookUpMat[X]
    h = torch.tanh(emb.view(-1,6) @ W1 + b1) ## activation function
    logits = h @ W2 + b2
    # -----------------
    norm = logits.exp()
    pred = norm / norm.sum(dim=1, keepdim=True)
    # ----------------- (SoftMax)
    loss = -pred[torch.arange(len(X)), Y].log().mean() # loss calculation
    
    print(loss)
    # gred zero
    for p in params:
        p.grad = None
    
    # backpropagartion
    loss.backward()

    # updating weights
    for p in params:
        p.data += -0.01 * p.grad

tensor(21.5311, grad_fn=<NegBackward0>)
tensor(20.1279, grad_fn=<NegBackward0>)
tensor(18.7688, grad_fn=<NegBackward0>)
tensor(17.4515, grad_fn=<NegBackward0>)
tensor(16.1375, grad_fn=<NegBackward0>)
tensor(14.8047, grad_fn=<NegBackward0>)
tensor(13.4486, grad_fn=<NegBackward0>)
tensor(12.0785, grad_fn=<NegBackward0>)
tensor(10.7213, grad_fn=<NegBackward0>)
tensor(9.4161, grad_fn=<NegBackward0>)
tensor(8.1834, grad_fn=<NegBackward0>)
tensor(7.0059, grad_fn=<NegBackward0>)
tensor(5.8721, grad_fn=<NegBackward0>)
tensor(4.8034, grad_fn=<NegBackward0>)
tensor(3.8559, grad_fn=<NegBackward0>)
tensor(3.0947, grad_fn=<NegBackward0>)
tensor(2.5344, grad_fn=<NegBackward0>)
tensor(2.1296, grad_fn=<NegBackward0>)
tensor(1.8223, grad_fn=<NegBackward0>)
tensor(1.5722, grad_fn=<NegBackward0>)
tensor(1.3571, grad_fn=<NegBackward0>)
tensor(1.1653, grad_fn=<NegBackward0>)
tensor(0.9909, grad_fn=<NegBackward0>)
tensor(0.8321, grad_fn=<NegBackward0>)
tensor(0.6892, grad_fn=<NegBackward0>)
tensor(0.5645, g